# Inter-Agent Communication (A2A)

The Agent-to-Agent (A2A) pattern enables independent AI agents to communicate, delegate tasks, and collaborate — regardless of the frameworks they're built with. A2A defines a standard: Agent Cards for discovery, typed Messages for communication, and HTTP/JSON-RPC for transport.

## Implementation with Flyte v2

This notebook reimplements the A2A calendar agent pattern from Chapter 15 using **Flyte v2 primitives**. Flyte's task protocol provides the same capabilities as A2A — typed inputs/outputs, discovery, secure credential injection — without an HTTP server.

#### A2A / ADK vs Flyte v2 — Key Differences

| Aspect | A2A / ADK | Flyte v2 |
|--------|-----------|----------|
| **Agent Card** | JSON file at `/.well-known/agent.json` | `TaskEnvironment(name=..., description=...)` |
| **Agent Discovery** | Well-known URI / curated registry | `flyte.remote.Task.get("team.agent_name")` |
| **Transport** | HTTP/JSON-RPC 2.0 over HTTPS | Flyte's gRPC task protocol |
| **Message format** | `Message(role, parts=[TextPart(...)])` | Typed Python dataclasses |
| **Authentication** | OAuth 2.0 / API key in HTTP headers | `flyte.Secret` + cluster RBAC |
| **Streaming updates** | Server-Sent Events (SSE) | `flyte.report.log.aio()` per chunk |
| **Task states** | submitted → working → completed | Flyte execution states in UI |
| **Artifacts** | `Artifact(parts=[...])` returned | Typed dataclass output — serialized by Flyte |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' openai

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret OPENAI_API_KEY --value sk-...

### 3. Import dependencies and configure TaskEnvironments

Each `TaskEnvironment` plays the role of an **Agent Card** in A2A — it declares the agent's name, capabilities (resources, image), and authentication requirements (secrets). In A2A, separate agents might run as distinct services; in Flyte v2, they run as separate task environments on the same cluster.

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass, field
from datetime import datetime, timedelta

from openai import AsyncOpenAI, OpenAI
import flyte

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="a2a-agents", python_version=(3, 12))
    .with_pip_packages("openai>=1.0.0")
)

_secrets = [flyte.Secret(key="OPENAI_API_KEY", as_env_var="OPENAI_API_KEY")]

# ── Agent Card equivalents (TaskEnvironments) ─────────────────────────────────

# calendar_agent: handles availability queries and scheduling
calendar_env = flyte.TaskEnvironment(
    name="calendar_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=_secrets,
)

# planner_agent: orchestrates other agents to complete complex requests
planner_env = flyte.TaskEnvironment(
    name="planner_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    secrets=_secrets,
)

# notifier_agent: sends confirmations and summaries
notifier_env = flyte.TaskEnvironment(
    name="notifier_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="512Mi"),
    secrets=_secrets,
)

### 4. Define the A2A message types

A2A defines `Message(role, parts=[TextPart | FilePart | DataPart])` as the communication unit between agents. Artifacts contain the actual results. In Flyte v2, typed dataclasses replace these JSON envelopes — same information, stronger type guarantees, and automatic serialization.

In [ ]:
@dataclass
class CalendarSlot:
    """A2A Artifact equivalent: structured availability slot."""
    date: str
    start_time: str
    end_time: str
    available: bool
    note: str = ""


@dataclass
class AvailabilityResult:
    """Output of calendar_agent — equivalent to A2A Task completion artifact."""
    request: str
    slots: list[CalendarSlot]
    summary: str


@dataclass
class ScheduledEvent:
    """Output of planner_agent after booking."""
    event_title: str
    attendees: list[str]
    slot: CalendarSlot
    confirmation_id: str


@dataclass
class NotificationResult:
    """Output of notifier_agent."""
    recipients: list[str]
    message: str
    sent: bool

### 5. Define the specialist agents (calendar + notifier)

In A2A, specialist agents expose HTTP endpoints and are discovered via Agent Cards. In Flyte v2, each specialist is a `@env.task` — discovered by name via `flyte.remote.Task.get()` and called directly. No HTTP server, no JSON-RPC, no port management.

In [ ]:
@calendar_env.task(
    retries=2,
    timeout=timedelta(minutes=2),
    cache="auto",
)
async def check_availability(
    user_name: str,
    date_range: str,
    duration_minutes: int = 60,
) -> AvailabilityResult:
    """
    Calendar agent: check a user's availability.

    In A2A this is an agent exposing a 'check_availability' skill via HTTP.
    In Flyte v2, it's a typed task — same interface, no server required.
    """
    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=512,
        messages=[{"role": "system", "content": f"You are a calendar assistant for {user_name}. " f"Today is {datetime.now().strftime('%Y-%m-%d')}. " "Generate 2-3 realistic available time slots for the requested date range. " "Format each slot as: DATE | HH:MM-HH:MM | available/busy | optional note. " "One slot per line, no other text."}] + [{
            "role": "user",
            "content": f"Date range: {date_range}\nMeeting duration: {duration_minutes} minutes",
        }],
    )

    slots: list[CalendarSlot] = []
    for line in response.choices[0].message.content.strip().splitlines():
        parts = [p.strip() for p in line.split("|")]
        if len(parts) >= 3:
            times = parts[1].split("-") if "-" in parts[1] else [parts[1], parts[1]]
            slots.append(CalendarSlot(
                date=parts[0],
                start_time=times[0].strip(),
                end_time=times[1].strip() if len(times) > 1 else times[0].strip(),
                available="available" in parts[2].lower(),
                note=parts[3] if len(parts) > 3 else "",
            ))

    available = [s for s in slots if s.available]
    summary = f"{len(available)} available slots found for {user_name} in {date_range}."

    return AvailabilityResult(request=f"{user_name} / {date_range}", slots=slots, summary=summary)


@notifier_env.task(retries=1, timeout=timedelta(minutes=1))
async def send_notification(
    recipients: list[str],
    event: ScheduledEvent,
) -> NotificationResult:
    """
    Notifier agent: compose and 'send' a meeting confirmation.
    In production, replace the return with an email/Slack API call.
    """
    client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": (
                f"Write a short meeting confirmation email for:\n"
                f"Event: {event.event_title}\n"
                f"Date: {event.slot.date} {event.slot.start_time}–{event.slot.end_time}\n"
                f"Attendees: {', '.join(event.attendees)}\n"
                f"Confirmation: {event.confirmation_id}\n"
                "Keep it to 3 sentences."
            ),
        }],
    )
    message = response.choices[0].message.content.strip()
    # In production: send via SendGrid / Slack / etc.
    return NotificationResult(recipients=recipients, message=message, sent=True)

### 6. Define the orchestrating planner agent

In A2A, a client agent discovers specialist agents via their Agent Cards and sends `sendTask` requests to their HTTP endpoints. In Flyte v2, the planner task calls specialist tasks directly via `await` — the same delegation pattern without HTTP overhead.

To reference a specialist agent deployed by another team, use `flyte.remote.Task.get("team.check_availability", auto_version="latest")` — the Flyte equivalent of A2A's agent discovery.

In [ ]:
import uuid


@planner_env.task(
    retries=1,
    timeout=timedelta(minutes=5),
    cache=flyte.Cache(behavior="disable"),
)
async def schedule_meeting(
    organizer: str,
    attendees: list[str],
    meeting_title: str,
    preferred_dates: str,
    duration_minutes: int = 60,
) -> ScheduledEvent:
    """
    Planner agent: orchestrates calendar + notifier agents to schedule a meeting.

    In A2A, this would:
      1. Discover calendar_agent via Agent Card
      2. Send sendTask to check_availability endpoint
      3. Discover notifier_agent and send sendTask for confirmation

    In Flyte v2, specialist agents are called directly via await — same delegation
    pattern, no HTTP server or JSON-RPC envelope required.

    To call a task from another team:
      remote_calendar = flyte.remote.Task.get("platform_team.check_availability",
                                               auto_version="latest")
      availability = await remote_calendar(user_name=organizer, ...)
    """
    import asyncio

    # ── Step 1: Check availability for all attendees in parallel ──────────────
    # A2A equivalent: send concurrent sendTask requests to calendar_agent
    all_users = [organizer] + attendees
    availability_tasks = [
        check_availability(
            user_name=user,
            date_range=preferred_dates,
            duration_minutes=duration_minutes,
        )
        for user in all_users
    ]
    all_availability = await asyncio.gather(*availability_tasks)

    # ── Step 2: Find a common available slot ──────────────────────────────────
    best_slot: CalendarSlot | None = None
    for avail in all_availability:
        for slot in avail.slots:
            if slot.available:
                best_slot = slot
                break
        if best_slot:
            break

    if best_slot is None:
        # Fallback: use first slot from organizer's calendar
        best_slot = all_availability[0].slots[0] if all_availability[0].slots else CalendarSlot(
            date=preferred_dates.split("-")[0].strip(),
            start_time="09:00", end_time="10:00", available=True,
        )

    confirmation_id = f"MTG-{uuid.uuid4().hex[:8].upper()}"
    event = ScheduledEvent(
        event_title=meeting_title,
        attendees=all_users,
        slot=best_slot,
        confirmation_id=confirmation_id,
    )

    # ── Step 3: Send notifications via notifier_agent ─────────────────────────
    # A2A equivalent: sendTask to notifier_agent endpoint
    await send_notification(recipients=all_users, event=event)

    return event

### 7. Run locally

In [ ]:
run = flyte.run(
    schedule_meeting,
    organizer="Alice",
    attendees=["Bob", "Carol"],
    meeting_title="Q3 Planning Session",
    preferred_dates="2025-08-11 to 2025-08-15",
    duration_minutes=90,
)
run.wait()
event: ScheduledEvent = run.outputs()[0]

print(f"Event: {event.event_title}")
print(f"Date: {event.slot.date} {event.slot.start_time}–{event.slot.end_time}")
print(f"Attendees: {event.attendees}")
print(f"Confirmation: {event.confirmation_id}")

### Running remotely

Each specialist agent (`calendar_env`, `notifier_env`) can be deployed independently and called via `flyte.remote.Task.get()` by any other team — the Flyte equivalent of A2A's agent registry and `sendTask` protocol. The `ScheduledEvent` dataclass is the artifact: structured, versioned, and inspectable in the UI without HTTP overhead.

In [ ]:
run = flyte.run(
    schedule_meeting,
    organizer="David",
    attendees=["Engineering Team"],
    meeting_title="Sprint Retrospective",
    preferred_dates="2025-08-18 to 2025-08-22",
    duration_minutes=60,
)
run.wait()
event = run.outputs()[0]
print(f"Scheduled: {event.event_title} on {event.slot.date} at {event.slot.start_time}")
print(f"Confirmation: {event.confirmation_id}")